# 🔄 Training Pipeline + Prediction Pipeline + FastAPI
### Malaria Outbreak Prediction — MLOps Project

---
## Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                     MLOps FLOW                              │
│                                                             │
│  feature_store/   →   train_pipeline.py                     │
│  (engineered data)     ├── Train LR + XGBoost               │
│                        ├── Compare on val AUC-ROC           │
│                        ├── Log both to MLflow               │
│                        └── Promote winner → Staging         │
│                                  ↓                          │
│                        predict_pipeline.py                  │
│                        ├── Load Staging model               │
│                        ├── Engineer features for new data   │
│                        ├── Scale + predict                  │
│                        └── Return probability + risk level  │
│                                  ↓                          │
│                        main.py  (FastAPI)                   │
│                        ├── POST /predict                    │
│                        ├── POST /predict/batch              │
│                        ├── GET  /model/info                 │
│                        └── POST /model/promote (Staging→Prod)│
└─────────────────────────────────────────────────────────────┘
```


---
## Part 1 — Run the Training Pipeline

The training pipeline is a **Python script** you run once.  
It trains both models, compares them, logs everything to MLflow,  
and automatically promotes the winner to **Staging**.


In [ ]:
# ── Step 1: Install dependencies ─────────────────────────────────────────────
!pip install xgboost mlflow scikit-learn pandas numpy fastapi uvicorn pydantic


In [ ]:
# ── Step 2: Run the training pipeline ────────────────────────────────────────
# This trains LR + XGBoost, logs to MLflow, promotes best to Staging
# You can also run this from terminal: python train_pipeline.py

import subprocess, sys
result = subprocess.run(
    [sys.executable, r"''' + BASE + r'''\src	rain_pipeline.py"],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


In [ ]:
# ── Step 3: Verify MLflow — check Staging model ──────────────────────────────
import mlflow
from mlflow import MlflowClient

MLFLOW_URI = r"file:///''' + MLRUNS.replace("\", "/") + r'''"
MODEL_NAME = "MalariaOutbreakPredictor"

mlflow.set_tracking_uri(MLFLOW_URI)
client = MlflowClient(tracking_uri=MLFLOW_URI)

print("All registered model versions:")
print(f"{'Version':<10} {'Stage':<15} {'Run ID':<36} {'Created'}")
print("-" * 80)
for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
    print(f"{v.version:<10} {v.current_stage:<15} {v.run_id:<36} "
          f"{v.creation_timestamp}")

print()
staging = client.get_latest_versions(MODEL_NAME, stages=["Staging"])
if staging:
    v = staging[0]
    print(f"✅ Current Staging model: version {v.version}  run_id={v.run_id}")
else:
    print("⚠️  No model in Staging yet. Run the training pipeline first.")


---
## Part 2 — Use the Prediction Pipeline

The prediction pipeline loads the Staging model and generates predictions.  
This can be used standalone (batch CSV) or imported by FastAPI.


In [ ]:
# ── Step 4: Load the prediction pipeline ─────────────────────────────────────
import sys
sys.path.append(r"''' + BASE + r'''\src")

from predict_pipeline import PredictionPipeline

# Loads model from MLflow Staging automatically
pipeline = PredictionPipeline(stage="Staging")
print(f"Model loaded  : {pipeline.metadata['best_model']}")
print(f"Stage         : {pipeline.stage}")
print(f"Threshold     : {pipeline.threshold}")
print(f"Features      : {len(pipeline.feature_cols)}")


In [ ]:
# ── Step 5: Single prediction ─────────────────────────────────────────────────
# Predict outbreak risk for Ghana 2023
# Providing 3 years of history enables lag feature computation

record = {
    "Country Name"             : "Ghana",
    "Country Code"             : "GHA",
    "Year"                     : 2023,
    "Malaria_Incidence"        : 180.5,
    "Precipitation_mm"         : 1200.0,
    "Pop_Density"              : 130.0,
    "GDP_per_Capita"           : 2200.0,
    "Temp_Annual_Mean_C"       : 26.5,
    "Temp_GrowingSeason_Mean_C": 27.1,
    # Prior years for lag features
    "_history": [
        {"Country Name": "Ghana", "Year": 2021,
         "Malaria_Incidence": 165.0, "Precipitation_mm": 1180.0,
         "Pop_Density": 126.0, "GDP_per_Capita": 2100.0,
         "Temp_Annual_Mean_C": 26.2, "Temp_GrowingSeason_Mean_C": 26.8},
        {"Country Name": "Ghana", "Year": 2022,
         "Malaria_Incidence": 172.0, "Precipitation_mm": 1195.0,
         "Pop_Density": 128.0, "GDP_per_Capita": 2150.0,
         "Temp_Annual_Mean_C": 26.4, "Temp_GrowingSeason_Mean_C": 27.0},
    ]
}

result = pipeline.predict_single(record)
import json
print(json.dumps(result, indent=2))


In [ ]:
# ── Step 6: Batch prediction from CSV ─────────────────────────────────────────
# You can run this on any new CSV with the same columns as the original dataset

import os
FEATURE_STORE = r"''' + FS + r'''"

# Use the test set as a demo batch
test_input  = os.path.join(FEATURE_STORE, "..", "malaria_final_dataset.csv")
test_output = os.path.join(FEATURE_STORE, "batch_predictions.csv")

df_results = pipeline.predict_batch(test_input, test_output)
print()
print("Sample predictions:")
print(df_results.head(10).to_string(index=False))
print()
print(f"Total outbreaks flagged : {df_results['outbreak_alert'].sum()}")
print(f"HIGH risk countries     : {(df_results['risk_level'] == 'HIGH').sum()}")


---
## Part 3 — Start the FastAPI Server

FastAPI loads the Staging model at startup and serves predictions via HTTP.


In [ ]:
# ── Step 7: Start FastAPI server ─────────────────────────────────────────────
# Run this in your TERMINAL (not in the notebook):
#
#   cd "C:\Users\Likhita Kolli\OneDrive - Hochschule Luzern\SEM_2\AI\Python_Project\files"
#   uvicorn main:app --reload --port 8000
#
# Then open:
#   Swagger UI → http://localhost:8000/docs
#   ReDoc      → http://localhost:8000/redoc
#   Health     → http://localhost:8000/

print("Start the server by running in your terminal:")
print()
print(r'  cd "C:\Users\Likhita Kolli\OneDrive - Hochschule Luzern\SEM_2\AI\Python_Project\files"')
print(r"  uvicorn main:app --reload --port 8000")
print()
print("Then open http://localhost:8000/docs")


In [ ]:
# ── Step 8: Test FastAPI with Python requests ─────────────────────────────────
# Run this AFTER the server is started in terminal

import requests, json

BASE_URL = "http://localhost:8000"

# ── Health check
r = requests.get(f"{BASE_URL}/")
print("Health check:", r.json())
print()

# ── Model info
r = requests.get(f"{BASE_URL}/model/info")
print("Model info:")
print(json.dumps(r.json(), indent=2))
print()

# ── Single prediction
payload = {
    "country_name"            : "Ghana",
    "country_code"            : "GHA",
    "year"                    : 2023,
    "malaria_incidence"       : 180.5,
    "precipitation_mm"        : 1200.0,
    "pop_density"             : 130.0,
    "gdp_per_capita"          : 2200.0,
    "temp_annual_mean_c"      : 26.5,
    "temp_growing_season_mean": 27.1,
}
r = requests.post(f"{BASE_URL}/predict", json=payload)
print("Single prediction:")
print(json.dumps(r.json(), indent=2))


In [ ]:
# ── Step 9: Promote Staging → Production (when ready) ─────────────────────────
# Only do this when you are satisfied with the Staging model's test metrics.
# This is the final MLOps gate before the model serves real traffic.

import requests, json

r = requests.post(
    "http://localhost:8000/model/promote",
    params={"secret_key": "malaria2024"}
)
print("Promote response:")
print(json.dumps(r.json(), indent=2))

# After promotion, verify in MLflow:
from mlflow import MlflowClient
import mlflow
mlflow.set_tracking_uri(r"file:///''' + MLRUNS.replace("\", "/") + r'''")
client = MlflowClient()
prod = client.get_latest_versions("MalariaOutbreakPredictor", stages=["Production"])
if prod:
    print(f"\n✅ Production model: v{prod[0].version}  run={prod[0].run_id}")


---
## ✅ Complete MLOps Flow Summary

```
train_pipeline.py          predict_pipeline.py        main.py (FastAPI)
─────────────────          ───────────────────        ─────────────────
Load feature_store    →    Load Staging model    →    POST /predict
Train LR + XGBoost         Engineer features          POST /predict/batch
Compare val AUC            Scale input                GET  /model/info
Log both to MLflow         Return probability         POST /model/promote
Promote winner to          + risk level                    ↓
  MLflow Staging                                      Staging → Production
```

### File structure
```
SEM_2/AI/
├── src/
│   ├── train_pipeline.py       ← run to retrain
│   ├── predict_pipeline.py     ← imported by FastAPI
│   └── main.py                 ← FastAPI app
├── models/
│   ├── logistic_regression.pkl
│   ├── xgboost_model.pkl
│   ├── scaler.pkl
│   └── model_metadata.json
├── mlruns/                     ← MLflow tracking data
└── Data/Processed/
    └── feature_store/          ← engineered splits
```

### MLflow stage transitions
```
None → Staging   (automatic, done by train_pipeline.py)
Staging → Production  (manual, via POST /model/promote or MLflow UI)
```
